# Ensemble & Test KL Divergence Evaluation
Loads multiple `submission.csv` files, averages their predicted probability distributions, and evaluates KL Divergence against `confident_test.csv` ground truth.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt


def write_csv_atomically(df, output_path, **to_csv_kwargs):
    output_path = Path(output_path)
    temp_path = output_path.with_name(f'.{output_path.name}.tmp')
    df.to_csv(temp_path, **to_csv_kwargs)
    temp_path.replace(output_path)


# ── Configuration ─────────────────────────────────────────────────────────
# Path to confident_test.csv (ground truth)
CONFIDENT_TEST_PATH = Path("/home/littl/data/data/confident_test.csv")

# List all submission CSV paths to ensemble
SUBMISSION_PATHS = [
    Path("/home/littl/ECE247A_Final_Project/ensemble/submission_baseline_AL.csv"),
    Path("/home/littl/ECE247A_Final_Project/ensemble/submission_spectrogram_gru_AL.csv"),
    Path("/home/littl/ECE247A_Final_Project/ensemble/JZ_conv1d_biGRU_confident_test_predictions_t12_ensemble.csv"),
    Path("/home/littl/ECE247A_Final_Project/ensemble/scalogram-0.9883_GC.csv"),
    Path("/home/littl/ECE247A_Final_Project/ensemble/spectrogram-0.8538_GC.csv"),
    Path("/home/littl/ECE247A_Final_Project/ensemble/JZ_vit_swin_confident_test_predictions_t8_ensemble.csv"),
    
    # add more as needed
]

num_submissions = len(SUBMISSION_PATHS)
# Class columns — must match submission file column names
CLASS_NAMES  = ["Seizure", "LPD", "GPD", "LRDA", "GRDA", "Other"]
TARGET_COLS  = [x.lower() + "_vote" for x in CLASS_NAMES]

# HQ consensus threshold (matches training/eval code)
HQ_THRESHOLD = 0.9

print(f"Target columns: {TARGET_COLS}")

In [ ]:
# ── Load ground truth from confident_test.csv ─────────────────────────────
test_df = pd.read_csv(CONFIDENT_TEST_PATH)

vote_cols = ["seizure_vote", "lpd_vote", "gpd_vote", "lrda_vote", "grda_vote", "other_vote"]
test_votes = test_df[vote_cols].values.astype(np.float64)
vote_sums  = test_votes.sum(axis=1, keepdims=True).clip(1.0)
targets    = test_votes / vote_sums          # soft label ground truth
consensus  = targets.max(axis=1)             # per-sample max agreement

print(f"Test samples  : {len(test_df)}")
print(f"HQ samples    : {(consensus >= HQ_THRESHOLD).sum()} (consensus >= {HQ_THRESHOLD})")
print(f"Consensus     : min={consensus.min():.3f}  median={np.median(consensus):.3f}  max={consensus.max():.3f}")

In [ ]:
# ── Load and validate submission files ────────────────────────────────────
all_preds = []

for path in SUBMISSION_PATHS:
    sub = pd.read_csv(path)
    assert "eeg_id" in sub.columns, f"Missing eeg_id in {path.name}"
    for col in TARGET_COLS:
        assert col in sub.columns, f"Missing column {col} in {path.name}"

    # Align to test_df order by eeg_id
    sub = sub.set_index("eeg_id").reindex(test_df["eeg_id"]).reset_index()
    assert not sub[TARGET_COLS].isnull().any().any(), f"NaNs after alignment in {path.name} — eeg_id mismatch?"

    preds = sub[TARGET_COLS].values.astype(np.float64)
    all_preds.append(preds)
    print(f"Loaded {path.name:40s} | shape={preds.shape} | row_sum min={preds.sum(axis=1).min():.4f}")

print(f"\nTotal submissions loaded: {len(all_preds)}")

In [ ]:
# ── Ensemble: average then renormalize ────────────────────────────────────
# ensemble = np.mean(all_preds, axis=0)                          # average across submissions
# ensemble = ensemble / ensemble.sum(axis=1, keepdims=True)      # renormalize to valid simplex

# ── Ensemble: weighted average then renormalize ───────────────────────────
# Weights correspond to each submission in SUBMISSION_PATHS order
# e.g. [1.0, 2.0] gives the second submission twice the weight of the first
WEIGHTS = [1.0] * num_submissions  # change these


assert len(WEIGHTS) == len(all_preds), "WEIGHTS length must match number of submissions"
w = np.array(WEIGHTS, dtype=np.float64) / sum(WEIGHTS)        # normalize weights to sum to 1
ensemble = np.sum([w[i] * all_preds[i] for i in range(len(all_preds))], axis=0)
ensemble = ensemble / ensemble.sum(axis=1, keepdims=True)      # renormalize to valid simplex

print(f"Weights applied: {dict(zip([p.name for p in SUBMISSION_PATHS], WEIGHTS))}")

print(f"Ensemble shape : {ensemble.shape}")
print(f"Row sum check  : min={ensemble.sum(axis=1).min():.6f}  max={ensemble.sum(axis=1).max():.6f}")

In [ ]:
# ── KL Divergence evaluation ───────────────────────────────────────────────
# KL(targets || ensemble): matches the convention used in training/evaluation code
kl_per_sample = (targets * np.log(targets.clip(1e-8) / ensemble.clip(1e-8))).sum(axis=1)

hq_mask = consensus >= HQ_THRESHOLD
true_labels = test_df["expert_consensus"].astype(str).to_numpy()
pred_label_idx = ensemble.argmax(axis=1)
pred_labels = np.array(CLASS_NAMES)[pred_label_idx]
pred_confidence = ensemble.max(axis=1)

kl_all = kl_per_sample.mean()
kl_hq  = kl_per_sample[hq_mask].mean() if hq_mask.any() else float("nan")

print("\n" + "="*50)
print("ENSEMBLE TEST KL DIVERGENCE")
print("="*50)
print(f"  ALL samples : {kl_all:.4f}  (n={len(kl_per_sample)})")
print(f"  HQ (>={HQ_THRESHOLD}) : {kl_hq:.4f}  (n={hq_mask.sum()}/{len(hq_mask)})")

In [ ]:
# ── Per-sample KL export ─────────────────────────────────────────────────
PER_SAMPLE_KL_PATH = Path("ensemble_per_sample_kl.csv")

metadata_cols = [
    col for col in ["eeg_id", "eeg_sub_id", "label_id", "patient_id", "expert_consensus"]
    if col in test_df.columns
]
per_sample_kl_df = test_df[metadata_cols].copy()
per_sample_kl_df["pred_label"] = pred_labels
per_sample_kl_df["target_confidence"] = consensus
per_sample_kl_df["pred_confidence"] = pred_confidence
per_sample_kl_df["kl_divergence"] = kl_per_sample
per_sample_kl_df["is_hq"] = hq_mask
per_sample_kl_df["is_correct"] = per_sample_kl_df["expert_consensus"] == per_sample_kl_df["pred_label"]

for idx, target_col in enumerate(TARGET_COLS):
    per_sample_kl_df[f"target_{target_col}"] = targets[:, idx]
    per_sample_kl_df[f"pred_{target_col}"] = ensemble[:, idx]

write_csv_atomically(per_sample_kl_df, PER_SAMPLE_KL_PATH, index=False)

preview_cols = [
    col for col in [
        "eeg_id",
        "expert_consensus",
        "pred_label",
        "target_confidence",
        "pred_confidence",
        "kl_divergence",
        "is_hq",
        "is_correct",
    ]
    if col in per_sample_kl_df.columns
]

print(f"Saved: {PER_SAMPLE_KL_PATH}  ({len(per_sample_kl_df)} rows)")
print("Highest-KL samples:")
print(
    per_sample_kl_df.sort_values("kl_divergence", ascending=False)[preview_cols]
    .head(10)
    .to_string(index=False)
)

In [ ]:
# ── Per-submission KL for comparison ──────────────────────────────────────
print(f"{'Submission':<45} {'KL ALL':>10} {'KL HQ':>10}")
print("-" * 67)

for path, preds in zip(SUBMISSION_PATHS, all_preds):
    preds_norm = preds / preds.sum(axis=1, keepdims=True)
    kl = (targets * np.log(targets.clip(1e-8) / preds_norm.clip(1e-8))).sum(axis=1)
    print(f"{path.name:<45} {kl.mean():>10.4f} {kl[hq_mask].mean():>10.4f}")

print("-" * 67)
print(f"{'ENSEMBLE':<45} {kl_all:>10.4f} {kl_hq:>10.4f}")

In [ ]:
# ── KL distribution plot ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# All samples
axes[0].hist(kl_per_sample, bins=60, color="steelblue", edgecolor="white", linewidth=0.4)
axes[0].axvline(kl_all, color="red", linestyle="--", label=f"Mean={kl_all:.4f}")
axes[0].set_title("KL Divergence — ALL samples")
axes[0].set_xlabel("KL Divergence"); axes[0].set_ylabel("Count")
axes[0].legend()

# HQ samples
axes[1].hist(kl_per_sample[hq_mask], bins=60, color="seagreen", edgecolor="white", linewidth=0.4)
axes[1].axvline(kl_hq, color="red", linestyle="--", label=f"Mean={kl_hq:.4f}")
axes[1].set_title(f"KL Divergence — HQ samples (consensus >= {HQ_THRESHOLD})")
axes[1].set_xlabel("KL Divergence"); axes[1].set_ylabel("Count")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Save ensemble submission ───────────────────────────────────────────────
OUTPUT_PATH = Path("submission_ensemble.csv")

pred_df = test_df[["eeg_id"]].copy()
pred_df[TARGET_COLS] = ensemble.tolist()
write_csv_atomically(pred_df, OUTPUT_PATH, index=False)

print(f"Saved: {OUTPUT_PATH}  ({len(pred_df)} rows)")
print(pred_df.head())

In [ ]:
# ── Confusion matrix export + plots ──────────────────────────────────────
CM_COUNTS_PATH = Path("ensemble_confusion_matrix_counts.csv")
CM_ROW_NORM_PATH = Path("ensemble_confusion_matrix_row_normalized.csv")

cm_counts = pd.crosstab(
    pd.Categorical(true_labels, categories=CLASS_NAMES, ordered=True),
    pd.Categorical(pred_labels, categories=CLASS_NAMES, ordered=True),
    rownames=["True label"],
    colnames=["Predicted label"],
    dropna=False,
)
cm_counts = cm_counts.reindex(index=CLASS_NAMES, columns=CLASS_NAMES, fill_value=0)
cm_row_norm = cm_counts.div(cm_counts.sum(axis=1).replace(0, np.nan), axis=0).fillna(0.0)

write_csv_atomically(cm_counts, CM_COUNTS_PATH)
write_csv_atomically(cm_row_norm, CM_ROW_NORM_PATH, float_format="%.6f")

def plot_confusion_matrix(ax, matrix_df, title, value_fmt):
    im = ax.imshow(matrix_df.values, cmap="Blues")
    ax.set_title(title)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_xticks(range(len(CLASS_NAMES)))
    ax.set_yticks(range(len(CLASS_NAMES)))
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
    ax.set_yticklabels(CLASS_NAMES)

    threshold = matrix_df.values.max() / 2 if matrix_df.values.size else 0
    for (row_idx, col_idx), value in np.ndenumerate(matrix_df.values):
        color = "white" if value > threshold else "black"
        ax.text(
            col_idx,
            row_idx,
            format(value, value_fmt),
            ha="center",
            va="center",
            color=color,
            fontsize=9,
        )

    return im

# Counts
fig1, ax1 = plt.subplots(figsize=(7, 6))
im_counts = plot_confusion_matrix(ax1, cm_counts, "Confusion Matrix (Counts)", ".0f")
fig1.colorbar(im_counts, ax=ax1, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

# Row-normalized
fig2, ax2 = plt.subplots(figsize=(7, 6))
im_norm = plot_confusion_matrix(ax2, cm_row_norm, "Confusion Matrix (Row-Normalized)", ".2f")
fig2.colorbar(im_norm, ax=ax2, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

print(f"Saved: {CM_COUNTS_PATH}")
print(f"Saved: {CM_ROW_NORM_PATH}")

In [ ]:
# ── confident_test expert_consensus totals ──────────────────────────────
expert_consensus_counts = (
    test_df["expert_consensus"]
    .value_counts()
    .reindex(CLASS_NAMES, fill_value=0)
)
expert_consensus_totals_df = pd.DataFrame({
    "class_name": expert_consensus_counts.index,
    "sample_count": expert_consensus_counts.values,
})
expert_consensus_totals_df["sample_fraction"] = (
    expert_consensus_totals_df["sample_count"] / expert_consensus_totals_df["sample_count"].sum()
)
expert_consensus_totals_df["sample_pct"] = 100 * expert_consensus_totals_df["sample_fraction"]

print("confident_test expert_consensus counts by class")
print(expert_consensus_totals_df.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print(f"\nTotal samples: {expert_consensus_totals_df['sample_count'].sum()}")
print(f"Expected if perfectly balanced: {len(test_df) / len(CLASS_NAMES):.6f} per class")